In [1]:
import pandas as pd
import numpy as np
from scipy.stats import weibull_min

# Nome do ficheiro
filename = 'hard_drive_failure_data.csv'

print("--- RELATÓRIO DE FIABILIDADE (PYTHON MLE) ---")

try:
    df = pd.read_csv(filename)
    
    # Função para tratar zeros (substitui 0 por 0.5)
    def tratar(dados):
        return np.where(dados <= 0, 0.5, dados)

    # Função auxiliar para calcular e imprimir resultados
    def analisar_fornecedor(nome, filtro):
        print(f"\n>>> FORNECEDOR: {nome}")
        # Aplica o filtro
        subset = df[filtro(df['model'])] #subset contém apenas as linhas de cada fornecedor
        
        if len(subset) > 0:
            lifetimes = tratar(subset['lifetime'].values)
            # Ajuste Weibull (MLE)
            beta, loc, eta = weibull_min.fit(lifetimes, floc=0)
            
            print(f"   Nº Falhas: {len(lifetimes)}")
            print(f"   Beta (Shape): {beta:.4f}")
            print(f"   Eta (Scale):  {eta:.4f} dias")
            
            # Pequena interpretação automática
            if beta < 0.9: diag = "Mortalidade Infantil (Decrescente)"
            elif beta > 1.1: diag = "Desgaste (Crescente)"
            else: diag = "Aleatório (Constante)"
            print(f"   Diagnóstico: {diag}")
        else:
            print("   Nenhum dado encontrado.")

    # 1. ALPHA (WDC)
    analisar_fornecedor("ALPHA (WDC)", lambda x: x.str.contains('WDC', na=False))

    # 2. BETA (ST) - Exclui HGST para não misturar
    analisar_fornecedor("BETA (ST)", lambda x: x.str.startswith('ST', na=False))

    # 3. GAMMA (Hitachi + HGST)
    # Nota: O fornecedor Gamma inclui marcas Hitachi e HGST
    analisar_fornecedor("GAMMA (Hitachi/HGST)", 
                        lambda x: x.str.contains('Hitachi', na=False) | x.str.contains('HGST', na=False))

except Exception as e:
    print(f"Erro: {e}")

--- RELATÓRIO DE FIABILIDADE (PYTHON MLE) ---

>>> FORNECEDOR: ALPHA (WDC)
   Nº Falhas: 151
   Beta (Shape): 1.2326
   Eta (Scale):  219.6794 dias
   Diagnóstico: Desgaste (Crescente)

>>> FORNECEDOR: BETA (ST)
   Nº Falhas: 2502
   Beta (Shape): 1.3310
   Eta (Scale):  352.6333 dias
   Diagnóstico: Desgaste (Crescente)

>>> FORNECEDOR: GAMMA (Hitachi/HGST)
   Nº Falhas: 318
   Beta (Shape): 0.9597
   Eta (Scale):  206.9199 dias
   Diagnóstico: Aleatório (Constante)
